In [2]:
import json
import pandas as pd
import re

def export_json_to_csv(json_path, csv_output_path):
    # 1. Đọc file JSON
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    rows = []
    for item in data:
        # Lấy nội dung comment và ngữ cảnh
        comment = item['data'].get('comment', '')
        context = item['data'].get('context', '')
        
        label = None
        note = ""
        
        # 2. Trích xuất Label và Note từ annotations
        if item.get('annotations'):
            # Lấy annotation đầu tiên
            ann = item['annotations'][0]
            for res in ann.get('result', []):
                # Nếu là nhãn (choices)
                if res.get('from_name') == 'label':
                    choices = res.get('value', {}).get('choices', [])
                    if choices:
                        # Dùng Regex lấy số đầu tiên (vd: "0 - Bình thường" -> 0)
                        match = re.search(r'\d', str(choices[0]))
                        if match:
                            label = int(match.group())
                
                # Nếu là ghi chú (textarea)
                if res.get('from_name') == 'note':
                    note_list = res.get('value', {}).get('text', [])
                    if note_list:
                        note = note_list[0]
        
        # Chỉ thêm vào danh sách nếu có nhãn (theo triết lý "không có khỏi gán")
        if label is not None:
            rows.append({
                'text': comment,
                'context': context,
                'label': label,
                'note': note
            })

    # 3. Tạo DataFrame và xuất CSV
    df = pd.DataFrame(rows)
    df.to_csv(csv_output_path, index=False, encoding='utf-8-sig')
    
    print(f"--- Hoàn tất ---")
    print(f"Đã xuất {len(df)} mẫu dữ liệu ra file: {csv_output_path}")

# Chạy script
export_json_to_csv('final_1k_thien_gold_sample.json', 'final_1k_output.csv')

--- Hoàn tất ---
Đã xuất 1123 mẫu dữ liệu ra file: final_1k_output.csv
